# Agentic Architecture & Orchestration



1. When should you start a new session with an injected structured summary rather than resuming a prior session with `--resume`?

A.Always — new sessions are always more reliable than resumed ones
**B.When the prior session's tool results are stale due to significant system state changes**
C.Only when the original session exceeded 100 turns
D.When the task domain changes from one programming language to another


Scenario: Multi-Agent Research System
Your coordinator agent delegates a broad research topic to four specialist subagents. The synthesis subagent's output has poor source attribution — it cannot identify which findings came from which source. What caused this and what is the fix?
<!-- A.The synthesis subagent needs a larger context window to hold all sources -->
B.The coordinator passed findings as plain concatenated text without preserving source metadata; fix by passing structured data that separates content from source URLs, titles, and page numbers
C.The web search subagent is not returning URLs in its results
D.Attribution requires a dedicated attribution subagent in the pipeline


A coordinator agent decomposes a research task into 3 fixed subtopics and consistently misses entire subject areas. What is the root cause?
A.The subagents are not returning their completed results back to the coordinator agent for aggregation
B.The context window is too small to handle all aspects of the full topic
<!-- C.Fixed decomposition is too narrow — the coordinator should adapt scope dynamically -->
D.The web search tool is returning incomplete or low-quality search results

During testing, combined outputs from the web search agent (85K tokens including page content) and the document analysis agent (70K tokens including reasoning chains) total 155K tokens, but the synthesis agent performs optimally with inputs under 50K tokens. What's the most effective solution?
A.Store findings in a vector database and give the synthesis agent retrieval tools to query during its work
B.Add an intermediate summarization agent that condenses findings before passing to synthesis
<!-- C.Modify upstream agents to return structured data (key facts, citations, relevance scores) instead of verbose content and reasoning -->
D.Have the synthesis agent process findings in sequential batches, maintaining running state between calls
✓ Correct — Explanation
Modifying upstream agents to return structured data addresses the root cause: the agents are returning far more token-volume than the synthesis step needs. Full page content and reasoning chains are intermediate artifacts of the search and analysis process — they are not the output the synthesis agent needs. Requiring agents to output key facts, citations, and relevance scores reduces token volume at the source while preserving the essential information. This is better than downstream approaches: vector retrieval (A) adds latency and requires the synthesis agent to know what to retrieve; intermediate summarization (B) adds another agent hop and may lose structured facts; sequential batches (D) require maintaining state across calls and do not reduce total token usage.


After a multi-agent research pipeline produces a comprehensive but shallow first draft, the coordinator needs to run iterative refinement. What is the architecturally correct approach?
A.Have the coordinator re-run the entire pipeline from scratch
B.Have the synthesis subagent expand all sections autonomously without coordinator guidance
<!-- C.The coordinator evaluates the synthesis for coverage gaps, re-delegates targeted queries to search/analysis subagents, and re-invokes synthesis until quality criteria are met -->
D.Ask the end user to identify which sections need more depth
✓ Correct — Explanation
Iterative refinement is a coordinator responsibility. The coordinator evaluates synthesis output against quality criteria (depth, coverage, citation density), identifies specific gaps (not vague dissatisfaction), formulates targeted follow-up queries, and re-delegates only what is needed. This loop continues until the coordinator's quality evaluation passes. Full pipeline re-runs are wasteful; user-driven gap identification is a design failure.


A subagent fails to retrieve data from a slow external API (timeout). What is the correct error-handling architecture?
A.Return an empty result set and let the coordinator decide what to do
B.Immediately propagate the error to the coordinator without any local recovery attempt
C.Attempt local recovery (e.g., retry); if unresolvable, propagate to coordinator with partial results and a structured error description
D.Log the error silently and continue as if the tool call succeeded
✓ Correct — Explanation
Subagents should attempt local recovery for transient errors (timeouts, temporary unavailability) before escalating. When escalating, they must return structured context: what data was retrieved before failure, what failed and why, and whether retry is appropriate. Silent error suppression prevents coordinator recovery; immediate escalation without retry wastes resources on recoverable failures.


You are designing a coordinator prompt for a multi-agent research system. Which prompt style leads to better subagent outcomes?
A.Step-by-step procedural instructions: "First search for X, then analyze Y, then synthesize Z"
B.Goal-oriented prompts specifying research goals and quality criteria rather than procedural steps
C.Single-sentence prompts to minimize token usage
D.Prompts that list which tools each subagent should use in what order
✗ Incorrect — Explanation
Goal-oriented coordinator prompts ("Research the impact of X on Y, ensure findings are cited, cover both short-term and long-term effects") outperform procedural step-by-step instructions. Specifying the *what* and *quality bar* rather than the *how* allows subagents to adapt their approach based on what they discover, producing more thorough and accurate results.


In an agentic loop, what does a `stop_reason` value of `"tool_use"` indicate?
A.Claude is requesting that a tool be executed and its result fed back
B.Claude has finished the task and no further action is needed
C.The agentic loop has exceeded its configured maximum iteration count limit and was terminated
D.Claude encountered an unrecoverable error and cannot continue
✗ Incorrect — Explanation
`stop_reason: "tool_use"` means Claude wants to invoke one or more tools. Your loop must execute those tools, append the results to the conversation history, and call Claude again. The loop only terminates when `stop_reason` is `"end_turn"` (or a budget/turn limit is hit).

An engineer asks your developer productivity agent to "add comprehensive tests to this legacy codebase." The agent immediately starts writing tests for the first file it finds, without understanding the codebase structure. What task decomposition pattern should the agent use?
A.Prompt chaining: write tests for each file in alphabetical order
B.Dynamic adaptive decomposition: first map codebase structure, identify high-impact areas, then create a prioritized plan that adapts as dependencies are discovered
C.Parallel decomposition: spawn one subagent per file simultaneously
D.Sequential fixed pipeline: analyze → write → run → fix
✓ Correct — Explanation
Open-ended tasks on unknown systems require dynamic adaptive decomposition. The agent cannot know the right approach until it understands the codebase. Phase 1: map the structure and understand the domain. Phase 2: identify high-impact, under-tested areas. Phase 3: create a prioritized plan that adapts as dependencies and test complexity emerge. This contrasts with prompt chaining, which requires knowing the steps upfront.


An interception hook blocks a `process_refund` call when the refund amount exceeds $500 and redirects to `escalate_to_human`. A colleague argues this logic should be in the system prompt instead. What is the strongest argument against the system prompt approach?
A.System prompts cannot reference specific dollar thresholds
B.Claude would ignore financial rules in system prompts
C.Hooks provide deterministic, guaranteed enforcement; prompt instructions are probabilistic and have a measurable failure rate on compliance-critical paths
D.System prompts are processed before tool definitions and cannot reference tool names
✓ Correct — Explanation
For business rule enforcement (refund thresholds, compliance gates), determinism is required. Hooks are code that executes deterministically — the rule fires every time without exception. A system prompt instruction might work 99% of the time, but for financial operations, 1% failure means real financial exposure. This is the fundamental programmatic vs prompt-based enforcement tradeoff.



`fork_session` is most appropriate when you want to:
A.Create independent exploration branches from a shared analysis baseline point
B.Continue a previously suspended session with updated and corrected context information
C.Run the same agent on multiple machines simultaneously for load balancing
D.Share session state between two different coordinator agents in a pipeline
✗ Incorrect — Explanation
`fork_session` creates parallel branches from a shared starting point — ideal when you want to explore two divergent approaches (e.g., two refactoring strategies, two testing architectures) without them interfering with each other. It is not for resuming sessions or distributed execution.



A customer support agent processes a refund request. The `process_refund` tool is called before `get_customer` has been invoked. Which root cause and fix apply?
A.Claude misunderstood the system prompt order; rewrite it with numbered steps
B.Missing programmatic prerequisite gate — `process_refund` should require a verified customer ID from a prior `get_customer` call in the current session to execute
C.The tools should be merged into a single `process_refund_with_lookup` tool
D.The agent should ask the user to confirm the customer ID before proceeding
✓ Correct — Explanation
This is the canonical prerequisite enforcement problem. The gate must be programmatic: before `process_refund` executes, the hook checks whether a verified customer ID exists in session state from a completed `get_customer` call. If not, the hook blocks the call and redirects to identity verification. Prompt-based ordering is insufficient for financial operations.



You use `--resume <session-name>` to continue a prior investigation. After resuming, you notice Claude is reasoning from stale file analysis done before you modified several files. What should you do?
A.Restart the entire session from scratch with no prior context at all
B.Use `fork_session` to create an independent branch from the stale point
C.Simply re-run the previous prompt — Claude will automatically detect and process file changes
D.Tell the resumed session which files changed so Claude re-analyzes them specifically
✗ Incorrect — Explanation
When resuming a session after code modifications, you must explicitly tell Claude which files changed. Claude will not automatically diff the file system against its prior analysis. Targeted re-analysis ("file X and Y were modified — please re-analyze them") is more efficient than full re-exploration and produces better results than relying on stale tool results.


A developer spent three hours analyzing a legacy codebase yesterday. Today they want to explore two different refactoring approaches starting from the same analysis baseline — without the approaches polluting each other. Which session management feature should they use?
A.--resume with yesterday's session name to continue from where they left off
B.Start two new sessions with the same system prompt and re-analyze each time
C./compact to reduce yesterday's session to a summary, then branch from there
D.fork_session to create two independent branches from the shared baseline
✗ Incorrect — Explanation
fork_session creates independent branches from an existing session's state — both branches share the analysis baseline but diverge independently from that point. Changes in branch A don't affect branch B. This is precisely the "explore two approaches from a common baseline" use case. --resume (A) would continue the single prior session, not create branches. Starting new sessions (C) requires expensive re-analysis. /compact reduces context but doesn't create forks.

A multi-agent research coordinator produces synthesis reports with poor coverage because it invokes all four subagents (web, document, synthesis, report) for every query regardless of complexity. What architectural improvement should be made?
A.Add a fifth "orchestration" subagent to manage the other four
B.Switch to a fixed sequential pipeline with no dynamic routing
C.Configure the coordinator to analyze query requirements and dynamically select only the relevant subagents
D.Increase the context window size to accommodate all four subagents' outputs
✓ Correct — Explanation
Always routing through the full pipeline is wasteful and can actually degrade quality (irrelevant subagents add noise). The coordinator should analyze each query and dynamically select only the subagents needed. A simple factual question might only need the web search subagent; a document-heavy analysis might skip web search entirely.

A developer implements an agentic loop that checks `response.content[0].text` to decide whether to stop. What is wrong with this approach?
A.The content array may be empty or missing, causing a runtime index error
B.Text content checks are unreliable — `stop_reason` is the authoritative signal
C.The developer should check `response.content[0].type` instead of the `.text` field
D.Nothing is wrong; this is a valid alternative approach to checking `stop_reason`
✓ Correct — Explanation
`stop_reason` is the correct and only reliable termination signal. Parsing text content (e.g., looking for "Done" or "Finished") is an anti-pattern: Claude's phrasing varies and the text may contain the word even mid-task. Always use `stop_reason === "end_turn"` to terminate the loop.


What is the role of a coordinator agent in a hub-and-spoke multi-agent architecture?
A.To execute all tool calls directly without delegating to others
B.To act as a proxy relay between subagents and external tool APIs
C.To decompose tasks, delegate to subagents, and aggregate results
D.To store a shared memory space accessible by all active subagents
✗ Incorrect — Explanation
In hub-and-spoke, the coordinator is the central hub. It decomposes the task, delegates to specialist subagents, routes information, and aggregates results. Subagents communicate only through the coordinator — never directly with each other.


When passing context from a web search subagent to a synthesis subagent, what should the coordinator include to preserve attribution?
A.Structured data separating content from metadata like source URLs and page numbers
B.Only the final text summary without source URLs; including metadata adds unnecessary token overhead cost
C.A plain text blob of all findings concatenated together without structure
D.A reference to the shared session ID where the subagent results are stored
✗ Incorrect — Explanation
The coordinator should use structured data to separate content (what was found) from metadata (where it came from — URLs, titles, page numbers). This allows the synthesis subagent to produce properly cited output and enables the coordinator to trace any finding back to its source for verification.

Scenario: You are building a customer support resolution agent using the Claude Agent SDK. The agent handles high-ambiguity requests like returns, billing disputes, and account issues. It has access to your backend systems through custom MCP tools (get_customer, lookup_order, process_refund, escalate_to_human). Your target is 80%+ first-contact resolution while knowing when to escalate.
Production data shows that in 12% of cases, your agent skips get_customer entirely and calls lookup_order using only the customer's stated name, occasionally leading to misidentified accounts and incorrect refunds. What change would most effectively address this reliability issue?
A.Add a programmatic prerequisite that blocks lookup_order and process_refund calls until get_customer has returned a verified customer ID
B.Enhance the system prompt to state that customer verification via get_customer is mandatory before any order operations
C.Add few-shot examples showing the agent always calling get_customer first, even when customers volunteer order details
D.Implement a routing classifier that analyzes each request and enables only the subset of tools appropriate for that request type
✓ Correct — Explanation
When a specific tool sequence is required for critical business logic (like verifying customer identity before processing refunds), programmatic enforcement provides deterministic guarantees that prompt-based approaches cannot. Options B and C rely on probabilistic LLM compliance, which is insufficient when errors have financial consequences. Option D addresses tool availability rather than tool ordering, which is not the actual problem.

You want to spawn multiple subagents to run in parallel. How do you achieve this with the Task tool?
A.Set `parallel: true` in the Task tool configuration object for concurrent execution
B.Call the Task tool separately, once per coordinator turn each time
C.Use the `fork_session` function instead of the standard Task tool
D.Emit multiple Task tool calls within a single coordinator response
✗ Incorrect — Explanation
Parallel subagent execution is triggered by emitting multiple Task tool calls in a single coordinator response (single turn). If you make separate Task calls across separate turns, they execute sequentially. The SDK interprets multiple tool calls in one response as concurrent invocations.


A coordinator agent needs to inventory all files matching a complex pattern across a large monorepo before deciding which ones to process. This discovery step generates thousands of lines of tool output. What is the recommended approach to prevent this verbose output from exhausting the coordinator's context budget?
A.Increase the coordinator's context window by switching to a model with a larger context limit
B.Have the coordinator run the discovery tools directly and use /compact after each batch
C.Spawn a dedicated Explore subagent to perform the discovery; the subagent's verbose tool results accumulate in its own isolated context and only a structured summary is returned to the coordinator
D.Run the discovery step synchronously before starting the agentic loop so output does not enter the conversation history
✓ Correct — Explanation
The Explore subagent pattern exists specifically for this problem: verbose discovery work runs inside the subagent's isolated context window. The coordinator only receives the structured summary (e.g., a list of matching file paths and their purposes) — not the raw tool output that generated it. This keeps the coordinator's context clean for high-level orchestration. Switching to a larger context model (A) delays but does not prevent exhaustion. /compact (B) compresses history but loses detail the coordinator may need later. Running discovery before the loop (D) still produces output that must live somewhere — it does not avoid the accumulation problem.


An agent has a named session from last week that contains extensive tool results from analyzing a codebase. Since then, 15 files have been modified and 3 new modules added. A developer wants to continue the investigation. What is the recommended approach?
A.Resume the old session with --resume; the model will detect changes automatically
B.Resume the old session and immediately inform the agent about which specific files changed, for targeted re-analysis
C.Start a new session and inject a structured summary of the prior session's key findings, then re-analyze only changed files
D.Resume the old session and run /compact first to condense the stale tool results
✗ Incorrect — Explanation
When prior tool results are stale (files have changed), starting fresh with injected summaries is more reliable than resuming. The old session's tool results reference code that no longer exists — the model may reason from outdated facts. A structured summary of key findings (architecture decisions, entry points, patterns) gives the new session the conceptual baseline without stale data. Option B (resume + inform) can work for minor changes but is risky when 15 files and 3 modules changed. Option A assumes automatic change detection, which doesn't exist. Option D doesn't help because /compact would just compress the stale data.


A customer contacts support with three separate issues: a billing error, a failed delivery, and a missing account credit. Your agent is handling them sequentially and missing interactions between them (the billing error and missing credit are related). What is the correct architectural approach?
A.Handle each issue in a separate session to avoid context contamination
B.Have the customer resubmit each issue separately
C.Decompose the multi-concern request into distinct items, investigate each in parallel using shared context, then synthesize a unified resolution
D.Use a larger context window to process all three issues together in a single prompt
✓ Correct — Explanation
Multi-concern requests should be decomposed into parallel investigation tracks that share context (so the billing/credit relationship is visible to both tracks), then synthesized into a unified resolution. Sequential handling misses cross-issue relationships. Separate sessions destroy the shared context needed to detect interactions. This is parallel decomposition with shared context — a core orchestration pattern.


After a coordinator spawns a subagent using the Task tool, does the subagent automatically have access to the coordinator's conversation history?
A.No — subagents have isolated context and receive information only via their prompt
B.Yes — the Agent SDK shares conversation context and tool results automatically between all agents
C.Yes — subagents inherit context through the SDK's shared session state store
D.Only if the coordinator explicitly sets `share_context: true` in the Task call
✗ Incorrect — Explanation
Subagents are isolated. They have no access to the coordinator's conversation history unless the coordinator explicitly includes the relevant information in the subagent's prompt. This is one of the most commonly tested facts on the exam.

Which mechanism must be included in the coordinator's `allowedTools` for it to spawn subagents?
A.Spawn
B.Task
C.Delegate
D.SubAgent
✓ Correct — Explanation
The `Task` tool is the Agent SDK mechanism for spawning subagents. If "Task" is not included in the coordinator's `allowedTools`, the coordinator cannot invoke subagents — the call will either fail or Claude will not attempt it.

Scenario: You are building a multi-agent research system using the Claude Agent SDK. A coordinator agent delegates to specialized subagents: one searches the web, one analyzes documents, one synthesizes findings, and one generates reports. The system researches topics and produces comprehensive, cited reports.
After running the system on the topic "impact of AI on creative industries," you observe that each subagent completes successfully — the web search agent finds relevant articles, the document analysis agent summarizes papers correctly, and the synthesis agent produces coherent output. However, the final reports cover only visual arts, completely missing music, writing, and film production. When you examine the coordinator's logs, you see it decomposed the topic into three subtasks: "AI in digital art creation," "AI in graphic design," and "AI in photography." What is the most likely root cause?
A.The synthesis agent lacks instructions for identifying coverage gaps in the findings it receives from other agents
B.The coordinator agent's task decomposition is too narrow, resulting in subagent assignments that don't cover all relevant domains of the topic
C.The web search agent's queries are not comprehensive enough and need to be expanded to cover more creative industry sectors
D.The document analysis agent is filtering out sources related to non-visual creative industries due to overly restrictive relevance criteria
✗ Incorrect — Explanation
The coordinator's logs reveal the root cause directly: it decomposed "creative industries" into only visual arts subtasks (digital art, graphic design, photography), completely omitting music, writing, and film. The subagents executed their assigned tasks correctly — the problem is what they were assigned. Options A, C, and D incorrectly blame downstream agents that are working correctly within their assigned scope.


Which SDK option caps the number of tool-use turns in an agentic loop?
A.max_iterations
B.turn_limit
C.stop_after
D.max_turns
✗ Incorrect — Explanation
`max_turns` (Python) / `maxTurns` (TypeScript) limits how many tool-use turns the loop can run. It counts tool-use turns only, not the final text response. `max_budget_usd` / `maxBudgetUsd` is the cost-based equivalent.

What does `stop_reason: "end_turn"` signal in an agentic loop?
A.A tool execution failed and must be retried by the loop handler
B.Claude has completed its response and the loop should terminate
C.The model ran out of available context window space mid-response
D.The user must provide additional input before the loop continues
✗ Incorrect — Explanation
`end_turn` means Claude has produced a final text-only response with no pending tool calls. This is the correct loop termination signal. Do not check text content or other heuristics to decide when to stop — always use `stop_reason`.

Scenario: Multi-Agent Research System
Your multi-agent research system takes 45 minutes to complete a comprehensive research report because the web search, document analysis, synthesis, and report generation subagents run sequentially. The coordinator waits for each to complete before starting the next. How do you reduce this to ~12 minutes without changing the synthesis step?
A.Use a faster model for each subagent
B.Emit parallel Task tool calls: web search and document analysis run simultaneously in a single coordinator response; synthesis waits for both to complete; report generation follows synthesis
C.Reduce the number of sources each subagent processes
D.Run all four subagents in parallel and merge the outputs
✓ Correct — Explanation
Web search and document analysis are independent — they do not need each other's results. Running them in parallel (single coordinator response with two Task calls) reduces their combined time from sum to max. Synthesis correctly waits for both (it needs both results). Report generation correctly follows synthesis. This is the parallel-then-sequential pattern: parallelize independent steps, sequence dependent ones. Running synthesis in parallel with earlier steps would break the dependency chain.


A coordinator agent is configured with allowedTools: ["Read", "Grep", "Glob"]. When it attempts to spawn a subagent by emitting a Task tool call, nothing happens — the call is silently ignored. What is the most likely cause?
A.The subagent's AgentDefinition is missing a required system prompt field
B."Task" is not included in the coordinator's allowedTools list configuration
C.Subagents can only be spawned from the root agent, not from nested coordinator agents
D.The Task tool call must include an allowedTools array for the subagent
✓ Correct — Explanation
The Task tool is the mechanism for spawning subagents. Like any other tool, it must be included in the coordinator's allowedTools configuration before it can be called. If "Task" is absent, the coordinator simply cannot emit Task calls. This is a common misconfiguration when assembling multi-agent systems: developers configure the subagents' AgentDefinitions carefully but forget to add "Task" to the coordinator's own allowed tools.

Scenario: Customer Support Resolution Agent
Your agent has resolved a complex billing dispute but cannot resolve a technical account issue that requires backend engineering access. It must hand off to a human agent. What should the handoff summary include?
A.A link to the conversation transcript only — the human agent can read it
B.Customer ID, root cause analysis, actions already taken, refund amounts applied, and recommended next action
C.A single sentence summary of the issue
D.The full raw conversation history in JSON format
✓ Correct — Explanation
Human agents receiving escalations typically do not have access to the full conversation transcript or AI session context. The structured handoff summary must be self-contained: customer ID (for lookup), root cause (what was determined), actions taken (what the AI already did, including any credits applied), and recommended next action (what the human should do first). This prevents duplicate actions and allows the human to work from a cold start.

What is the key difference between `prompt chaining` and `dynamic adaptive decomposition` for task execution?
A.Prompt chaining uses multiple models; dynamic decomposition uses one
B.Prompt chaining is for sequential predictable steps; dynamic decomposition generates subtasks based on intermediate findings
C.Dynamic decomposition requires a coordinator agent; prompt chaining can run in a single agent
D.Prompt chaining is cheaper; dynamic decomposition produces better results but at higher cost
✓ Correct — Explanation
Prompt chaining is best for workflows with known, fixed steps (e.g., file-by-file analysis → cross-file synthesis). Dynamic adaptive decomposition is for open-ended tasks where the full scope is unknown upfront — the plan adapts as findings emerge. A legacy codebase test coverage task uses dynamic decomposition because you don't know the structure or high-impact areas until you start exploring.

Scenario: Customer Support Resolution Agent
Your customer support agent is resolving billing disputes 80% of the time on first contact, but 6% of transactions are processed without the mandatory identity verification step. You have a clear system prompt instruction: "Always verify customer identity before processing any transaction." What is the most appropriate fix?
A.Strengthen the system prompt instruction with more explicit language and examples
B.Add few-shot examples demonstrating correct identity-first sequences
C.Implement a programmatic prerequisite gate: block `process_refund` and `process_payment` tool calls until `get_customer` has returned a verified customer ID in the current session
D.Add a classifier that detects when Claude skips verification and triggers a retry
✓ Correct — Explanation
System prompt instructions and few-shot examples address probabilistic behavior — they reduce but cannot eliminate the failure rate. A 6% skip rate on financial identity verification is a compliance failure. Programmatic prerequisite gates (hooks that block payment tools until verification returns a valid ID) provide deterministic, guaranteed enforcement. The fix must be architectural, not prompt-based.


You are choosing a decomposition strategy for two workflows: (A) a code review that must check security, performance, and style for each file in a PR; (B) an investigation into why production errors spiked — the cause is unknown and each finding may redirect the investigation. Which strategy fits each?
A.Dynamic decomposition for both: uncertainty always favors adaptive plans
B.Prompt chaining for A, dynamic decomposition for B
C.Prompt chaining for both: structured steps produce more consistent output
D.Dynamic decomposition for A, prompt chaining for B
✗ Incorrect — Explanation
Code review (A) has a predictable multi-aspect structure: the same three checks apply to every file, making prompt chaining (fixed sequential passes) ideal. Error investigation (B) is open-ended: what you find in step one determines what to look at in step two. Dynamic decomposition — generating subtasks based on intermediate findings — is the right fit. Applying dynamic decomposition to the code review wastes overhead for no gain; applying prompt chaining to the investigation would miss the adaptive branching the task requires.



A PostToolUse hook is best used to accomplish which of the following?
A.Block a tool from being called at all during the current session
B.Override Claude's decision about which tool to invoke in the next step of the agentic loop
C.Normalize heterogeneous data formats like timestamps before Claude processes them
D.Log all tool calls and their results to an external database for auditing
✓ Correct — Explanation
PostToolUse hooks intercept tool results *after* execution but *before* Claude processes them. They are ideal for normalizing heterogeneous data — e.g., converting Unix timestamps, ISO 8601 strings, and numeric codes from different MCP tools into a consistent format. Blocking tool calls before execution requires a pre-call interception hook.






A coordinator agent needs to gather data from three independent sources (web, database, document store) before synthesizing a final answer. Currently it spawns each subagent in a separate turn — web search, then wait, then database, then wait, then document store, then wait — taking ~90 seconds total. How should you modify the coordinator to reduce latency?
A.Use fork_session so each subagent runs in its own isolated session
B.Emit all three Task tool calls in a single coordinator response; they will execute in parallel
C.Configure the coordinator's max_tokens higher so it can process all three in one reasoning pass
D.Chain the subagents so each passes its results directly to the next without returning to the coordinator
✗ Incorrect — Explanation
Multiple Task tool calls emitted in a single coordinator response execute in parallel — this is the designed mechanism for concurrent subagent execution. By returning all three Task calls at once (instead of one per turn), all three subagents start simultaneously. Total latency becomes max(web, database, docs) rather than sum. Fork_session (A) is for exploring divergent approaches from a shared baseline, not for parallelism. Max_tokens (C) controls output length, not execution model. Chaining subagents (D) destroys parallelism by creating sequential dependencies.



A multi-phase coordinator agent crashes after completing phase 2 of a 4-phase research pipeline. The pipeline must resume without restarting from the beginning. Which architecture enables reliable crash recovery?
A.Rely on `--resume` to reload the conversation history and infer progress from prior messages
B.Implement session checkpointing with `fork_session` at the end of each phase
C.Have each phase export a structured state manifest (completed work, partial results, next phase inputs) to a known location, and have the coordinator load that manifest on startup to determine where to resume
D.Configure the coordinator with a high `max_tokens` budget so phases are less likely to be interrupted
✓ Correct — Explanation
`--resume` reloads conversation history but cannot recover in-progress subagent state or structured phase outputs — it tells you what was discussed, not what was computed. `fork_session` creates a branch of the conversation but also does not persist phase outputs across crashes. The correct pattern is structured state exports: at the end of each phase, the coordinator writes a manifest (phase number, completed agent outputs, next-phase inputs) to a known file location. On startup, the coordinator reads the manifest to determine the last completed phase and continues from there. High `max_tokens` budgets (D) affect generation length, not crash recovery.



A financial workflow requires that `verify_identity` must always run before `process_payment`. A senior engineer proposes adding a clear system prompt instruction: "Always call verify_identity before process_payment." Why is this insufficient for a production financial system?
A.System prompt instructions are ignored when tools are present
B.Claude cannot read multi-sentence system prompts reliably
C.Prompt instructions have a non-zero failure rate; programmatic prerequisites with hooks provide deterministic enforcement
D.The instruction would only apply during the first tool call, not subsequent calls
✓ Correct — Explanation
Prompt instructions influence Claude probabilistically — they work most of the time but have a measurable failure rate. For compliance-critical financial operations (identity verification before payment), "most of the time" is not acceptable. Programmatic prerequisites — hooks that literally block `process_payment` until `verify_identity` has returned a verified customer ID — provide guaranteed enforcement regardless of Claude's reasoning.



A large code review task is producing inconsistent results because Claude is missing interactions between files analyzed separately. What decomposition strategy addresses this?
A.Use a larger model with a bigger context window to process all files simultaneously
B.Split into per-file local analysis passes followed by a separate cross-file integration pass
C.Have Claude re-read the entire codebase after each file analysis
D.Increase max_turns to allow more iterations
✓ Correct — Explanation
Attention dilution occurs when too many files are processed simultaneously. The correct pattern: (1) run focused per-file analysis passes where each file gets full attention, then (2) run a dedicated cross-file integration pass with summaries from step 1. This two-phase approach reliably surfaces cross-file issues without attention degradation.






